# 05 · Evaluación en test

**Pregunta de este notebook.** ¿Se sostiene la hipótesis H en personas que no intervinieron en ninguna decisión? Se aplican a los 20 registros de test (10 personas) el detector y los parámetros fijados en desarrollo (`docs/parametros_test.json`), y se miden las tres predicciones con el criterio escrito antes de abrir test (`docs/objetivos_hipotesis.md`):

| Predicción | Qué se mide | Se cumple si |
|---|---|---|
| **P1**, dentro de la noche | AUC dentro de la noche de P_RR, en cada registro con al menos 20 minutos de cada tipo | la media es > 0,5 y su IC 95 % (bootstrap por persona) no incluye 0,5 |
| **P2**, entre noches | Spearman entre el nivel nocturno de la EDR y el % de minutos con apnea de cada registro | ρ > 0 y su IC 95 % no incluye 0 |
| **P3**, detección | AUC de C con todos los minutos; sensibilidad, especificidad y exactitud con el umbral fijo | AUC > 0,5 y su IC 95 % no incluye 0,5 |

**H se sostiene si se cumplen las tres.**

**Reglas.**
- En este notebook no se ajusta nada con test: ni la banda, ni la ventana, ni los parámetros de C, ni el umbral, ni los cortes de confianza. Se leen del archivo y se verifica que el archivo es el mismo que dejó `04`.
- Se corre **una sola vez**. Todo el análisis (predicciones, preguntas secundarias y figuras) se escribió antes de correrlo y se probó en **modo ensayo**, con registros de desarrollo en lugar de test. La versión sin ejecutar quedó registrada en el repositorio antes de la ejecución.
- Los umbrales de F y del RR solo, que se usan solo en las preguntas secundarias, se fijan con desarrollo por Youden, igual que el de C.
- Los intervalos de confianza se calculan remuestreando **personas** (4000 veces), porque los minutos y las noches de una misma persona no son independientes. Con 10 personas, los intervalos son anchos.

In [ ]:
import hashlib
import json
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

from src.config import SPLIT_CSV
from src.features.detector import c_score, confidence, evaluable, load_params, minute_table, youden_threshold
from src.viz.plots import ANCHO, APNEA, COLORES_CAT, GRIS, NORMAL, save, setup

setup()
RUTA_PARAMS = ROOT / "docs" / "parametros_test.json"
SHA256_PARAMS = "d2aa704837aa6543b3403fb5c11cade858c310da5d2cf7f25b6990ac503271c4"   # parametros_test.json del commit 41a73f0
assert hashlib.sha256(RUTA_PARAMS.read_bytes()).hexdigest() == SHA256_PARAMS, "los parámetros no son los fijados en 04"
P = load_params(RUTA_PARAMS)

split = pd.read_csv(SPLIT_CSV)
SUJETO, CATEG = split.set_index("record").subject, split.set_index("record").category
DEV = split[split.set == "dev"].record.tolist()
# Modo ensayo: prueba el código con 10 personas de desarrollo en lugar de test, sin abrir test.
ENSAYO = os.environ.get("NB05_ENSAYO") == "1"
if ENSAYO:
    personas = sorted(split[split.set == "dev"].subject.unique())[:10]
    TEST = split[split.subject.isin(personas)].record.tolist()
else:
    TEST = split[split.set == "test"].record.tolist()
guardar = (lambda fig, nombre: None) if ENSAYO else save
print(("MODO ENSAYO (desarrollo): " if ENSAYO else "TEST: ") + f"{len(TEST)} registros de {SUJETO[TEST].nunique()} personas")
print(json.dumps({k: P[k] for k in ("mu1", "s1", "mu2", "s2", "w", "umbral", "cortes_confianza_mov_rel")}, indent=1))

## 1. La tabla por minuto

Para cada registro se corre exactamente el pipeline de desarrollo (`src/features/detector.py`, `minute_table`): filtro, detector de latidos propio, limpieza del RR, EDR con Wiener, espectros de Welch en 240 s y nivel nocturno de la EDR. Después se aplican los parámetros fijos: C, la decisión y la confianza. Se calcula lo mismo en desarrollo, para fijar los umbrales de F y del RR solo y para comparar.

In [ ]:
def aplicar(tab):
    tab = tab.assign(subject=tab.record.map(SUJETO), category=tab.record.map(CATEG))
    tab["C"] = c_score(tab, P)
    tab["decision"] = (tab.C > P["umbral"]).astype(int)
    tab["confianza"] = confidence(tab.mov_rel, tuple(P["cortes_confianza_mov_rel"]))
    return tab

with ProcessPoolExecutor(4) as ex:
    T = aplicar(pd.concat(list(ex.map(minute_table, TEST)), ignore_index=True))
    D = aplicar(pd.concat(list(ex.map(minute_table, DEV)), ignore_index=True))
COLS = ["P_RR", "P_EDR", "F", "C"]
et = T[evaluable(T)].dropna(subset=COLS).reset_index(drop=True)
ed = D[evaluable(D)].dropna(subset=COLS).reset_index(drop=True)
y_t, y_d = et.label.to_numpy(), ed.label.to_numpy()

U = {"C": P["umbral"], "F": youden_threshold(ed.F.to_numpy(), y_d), "RR solo": youden_threshold(ed.P_RR.to_numpy(), y_d)}
COL = {"C": "C", "F": "F", "RR solo": "P_RR"}

resumen = (T.groupby("record").agg(persona=("subject", "first"), categoria=("category", "first"), minutos=("minute", "size"))
           .join(et.groupby("record").agg(evaluados=("label", "size"), apnea=("label", "mean"))))
resumen["apnea"] = 100 * resumen.apnea
print(f"minutos evaluados: {len(et)} de {len(T)} ({100 * len(et) / len(T):.1f} %), con apnea: {100 * y_t.mean():.1f} %")
print("umbrales fijados en desarrollo:", {k: round(v, 3) for k, v in U.items()})
resumen.round(1)

## 2. Las predicciones

In [ ]:
def auc(x, y):
    x, y = np.asarray(x, float), np.asarray(y, int)
    r = rankdata(x); n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

rng = np.random.default_rng(2026)
N_BOOT = 4000

def sorteos(personas, n=N_BOOT):
    """Remuestreos de personas con reposición (la misma lista para todas las medidas: comparaciones pareadas)."""
    personas = np.unique(personas)
    return [rng.choice(personas, len(personas)) for _ in range(n)]

def indices(suj, sorteo):
    return np.concatenate([np.flatnonzero(suj == u) for u in sorteo])

def ic(boot):
    return np.nanpercentile(boot, [2.5, 97.5])

# P1: AUC dentro de la noche de P_RR
dentro_t = pd.DataFrame([{"record": r, "subject": SUJETO[r], "category": CATEG[r],
                          "AUC RR": auc(g.P_RR, g.label), "AUC F": auc(g.F, g.label)}
                         for r, g in et.groupby("record") if (g.label == 1).sum() >= 20 and (g.label == 0).sum() >= 20])
dentro_d = pd.DataFrame([{"record": r, "AUC RR": auc(g.P_RR, g.label), "AUC F": auc(g.F, g.label)}
                         for r, g in ed.groupby("record") if (g.label == 1).sum() >= 20 and (g.label == 0).sum() >= 20])
s1 = dentro_t.subject.to_numpy()
b1 = np.array([dentro_t["AUC RR"].to_numpy()[indices(s1, so)].mean() for so in sorteos(s1)])

# P2: nivel nocturno de la EDR contra el % de apnea
niv_t = et.groupby("record").agg(EDR_noche=("EDR_noche", "first"), frac=("label", "mean"), subject=("subject", "first"),
                                 category=("category", "first"))
niv_d = ed.groupby("record").agg(EDR_noche=("EDR_noche", "first"), frac=("label", "mean"), category=("category", "first"))
s2 = niv_t.subject.to_numpy()
b2 = np.array([spearmanr(niv_t.EDR_noche.to_numpy()[i], niv_t.frac.to_numpy()[i])[0]
               for i in (indices(s2, so) for so in sorteos(s2))])

# P3: C con todos los minutos y el umbral fijo
def metricas(y, pred):
    tp = ((pred == 1) & (y == 1)).sum(); tn = ((pred == 0) & (y == 0)).sum()
    return {"sensibilidad": tp / (y == 1).sum(), "especificidad": tn / (y == 0).sum(), "exactitud": (tp + tn) / len(y)}

s3 = et.subject.to_numpy()
SORTEOS_MIN = sorteos(s3)
IDX_MIN = [indices(s3, so) for so in SORTEOS_MIN]
c_t, d_t = et.C.to_numpy(), et.decision.to_numpy()
b3 = pd.DataFrame([{"AUC": auc(c_t[i], y_t[i]), **metricas(y_t[i], d_t[i])} for i in IDX_MIN])

m1, m2, m3 = dentro_t["AUC RR"].mean(), spearmanr(niv_t.EDR_noche, niv_t.frac)[0], auc(c_t, y_t)
(l1, h1), (l2, h2), (l3, h3) = ic(b1), ic(b2), ic(b3.AUC)
veredicto = pd.DataFrame([
    {"predicción": "P1 · AUC dentro de la noche del RR (media)", "n": f"{len(dentro_t)} noches, {dentro_t.subject.nunique()} personas",
     "valor": m1, "IC 95 %": f"{l1:.3f} a {h1:.3f}", "criterio": "> 0,5 y el IC no incluye 0,5", "se cumple": m1 > 0.5 and l1 > 0.5},
    {"predicción": "P2 · ρ entre el nivel EDR de la noche y el % de apnea", "n": f"{len(niv_t)} noches, {niv_t.subject.nunique()} personas",
     "valor": m2, "IC 95 %": f"{l2:.3f} a {h2:.3f}", "criterio": "> 0 y el IC no incluye 0", "se cumple": m2 > 0 and l2 > 0},
    {"predicción": "P3 · AUC de C con todos los minutos", "n": f"{len(et)} minutos, {et.subject.nunique()} personas",
     "valor": m3, "IC 95 %": f"{l3:.3f} a {h3:.3f}", "criterio": "> 0,5 y el IC no incluye 0,5", "se cumple": m3 > 0.5 and l3 > 0.5},
]).set_index("predicción")
H_SOSTENIDA = bool(veredicto["se cumple"].all())
print("H se sostiene" if H_SOSTENIDA else "H NO se sostiene", "(se cumplen", int(veredicto["se cumple"].sum()), "de 3 predicciones)")
veredicto.round(3)

Con el umbral fijado en desarrollo, la clasificación de los minutos de test:

In [ ]:
op = metricas(y_t, d_t)
pd.DataFrame([{"medida": k, "test": v, "IC 95 %": "{:.3f} a {:.3f}".format(*ic(b3[k]))} for k, v in op.items()]).set_index("medida").round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(ANCHO, 2.4))
# P1
ax[0].scatter(np.full(len(dentro_d), 0), dentro_d["AUC RR"], s=8, color=GRIS, label="desarrollo")
ax[0].scatter(np.full(len(dentro_t), 1), dentro_t["AUC RR"], s=10, color="black", label="test")
ax[0].errorbar(1.25, m1, yerr=[[m1 - l1], [h1 - m1]], fmt="o", color=APNEA, ms=4, capsize=3)
ax[0].axhline(0.5, color=GRIS, ls=":", lw=0.8)
ax[0].set(xticks=[0, 1], xticklabels=["desarrollo", "test"], xlim=(-0.5, 1.6), ylim=(0.3, 1.02),
          ylabel="AUC dentro de la noche", title="P1 · RR dentro de la noche")
# P2
ax[1].scatter(niv_d.EDR_noche, 100 * niv_d.frac, s=8, color=GRIS, label="desarrollo")
ax[1].scatter(niv_t.EDR_noche, 100 * niv_t.frac, s=12, c=[COLORES_CAT[c] for c in niv_t.category], label="test")
ax[1].set(xlabel="nivel nocturno de la EDR", ylabel="% de minutos con apnea", title=f"P2 · ρ = {m2:.2f}".replace(".", ","))
# P3
o_ = np.argsort(-c_t); yy = y_t[o_]
ax[2].plot(np.r_[0, np.cumsum(1 - yy) / (1 - yy).sum()], np.r_[0, np.cumsum(yy) / yy.sum()], color="black", lw=1)
ax[2].plot(1 - op["especificidad"], op["sensibilidad"], "o", color=APNEA, ms=5)
ax[2].plot([0, 1], [0, 1], ":", color=GRIS, lw=0.8)
ax[2].set(xlabel="1 − especificidad", ylabel="sensibilidad", title=f"P3 · AUC de C = {m3:.2f}".replace(".", ","), aspect="equal")
fig.tight_layout()
guardar(fig, "05_predicciones")

**Cómo leer la figura.** Izquierda: el AUC dentro de cada noche de test (negro) junto a los de desarrollo (gris), con la media de test y su intervalo (naranja). Centro: cada noche de test coloreada por categoría (A, B, C) sobre las de desarrollo. Derecha: la curva ROC de C en test, con el punto del umbral fijado en desarrollo.

## 3. Preguntas secundarias

### 3.1 ¿C mejora a F y al RR solo?

Las tres features, cada una con su umbral de Youden de desarrollo, en las mismas medidas que la comparación de `04` (sección 8). Las diferencias C − F y C − RR solo tienen intervalos de un bootstrap **pareado** de personas: en cada repetición, las tres features se evalúan con los mismos minutos y noches.

In [ ]:
def medidas(tab, feat, umbral):
    """Las medidas de la comparación para una feature, en los minutos de `tab`."""
    x, y = tab[COL[feat]].to_numpy(), tab.label.to_numpy()
    pred = (x > umbral).astype(int)
    aucs_n = [auc(g_[COL[feat]], g_.label) for _, g_ in tab.groupby("record")
              if (g_.label == 1).sum() >= 20 and (g_.label == 0).sum() >= 20]
    nn = tab.assign(p=pred).groupby("record").agg(real=("label", "mean"), est=("p", "mean"), categ=("category", "first"))
    return {**metricas(y, pred), "AUC (todos los minutos)": auc(x, y),
            "AUC dentro de la noche (media)": np.mean(aucs_n) if aucs_n else np.nan,
            "ρ por noche": spearmanr(nn.real, nn.est)[0] if len(nn) > 2 else np.nan,
            "% marcado en controles": 100 * nn[nn.categ == "C"].est.mean() if (nn.categ == "C").any() else np.nan}

def medidas_remuestreo(feat):
    """Las medidas en cada remuestreo de personas. Cada persona sorteada entra con todos sus registros
    (renombrados para que una persona sorteada dos veces cuente como dos noches)."""
    out = []
    for so in SORTEOS_MIN[:2000]:
        partes = [et[et.subject == u].assign(record=lambda d_, k=k: d_.record + f"#{k}") for k, u in enumerate(so)]
        out.append(medidas(pd.concat(partes, ignore_index=True), feat, U[feat]))
    return pd.DataFrame(out)

punto = {f_: medidas(et, f_, U[f_]) for f_ in COL}
boot_f = {f_: medidas_remuestreo(f_) for f_ in COL}
tabla_feat = pd.DataFrame(punto).T
filas = []
for otra in ("F", "RR solo"):
    fila = {"comparación": f"C − {otra}"}
    for m_ in tabla_feat.columns:
        d_ = (boot_f["C"][m_] - boot_f[otra][m_]).to_numpy()
        dif_p = punto["C"][m_] - punto[otra][m_]
        if np.isnan(dif_p):
            fila[m_] = "—"
            continue
        lo, hi = ic(d_)
        if np.allclose(d_, 0):
            fila[m_] = "0 (idénticas)"       # dentro de la noche, C ordena los minutos igual que el RR
            continue
        fila[m_] = f"{dif_p:+.3f} ({lo:+.3f} a {hi:+.3f}){'*' if lo > 0 or hi < 0 else ''}"
    filas.append(fila)
dif_feat = pd.DataFrame(filas).set_index("comparación")
tabla_feat.round(3)

In [ ]:
dif_feat

Las diferencias marcadas con \* tienen un intervalo que no incluye el cero. Para el % marcado en controles, una diferencia negativa favorece a C.

### 3.2 ¿Sirve la confianza?

La exactitud del detector según la confianza de cada minuto, que depende del movimiento con los cortes de desarrollo.

In [ ]:
filas = []
for c_ in ("alta", "media", "baja"):
    sel_t, sel_d = (et.confianza == c_).to_numpy(), (ed.confianza == c_).to_numpy()
    bt = [((d_t[i] == y_t[i])[sel_t[i]]).mean() if sel_t[i].any() else np.nan for i in IDX_MIN[:2000]]
    filas.append({"confianza": c_, "% de minutos (test)": 100 * sel_t.mean(),
                  "exactitud (desarrollo)": (ed.decision.to_numpy()[sel_d] == y_d[sel_d]).mean(),
                  "exactitud (test)": (d_t[sel_t] == y_t[sel_t]).mean() if sel_t.any() else np.nan,
                  "IC 95 % (test)": "{:.3f} a {:.3f}".format(*ic(bt))})
conf_tab = pd.DataFrame(filas).set_index("confianza")
conf_tab.round(3)

### 3.3 Por noche

El % de minutos marcados como apnea en cada noche contra el % real, y los minutos marcados en los controles (categoría C, sin apnea).

In [ ]:
noches_t = (et.assign(ok=(et.decision == et.label)).groupby("record")
            .agg(real=("label", "mean"), estimada=("decision", "mean"), exactitud=("ok", "mean"),
                 categoria=("category", "first"), persona=("subject", "first")))
noches_t[["real", "estimada"]] *= 100
noches_d = ed.groupby("record").agg(real=("label", "mean"), estimada=("decision", "mean"))
s_n = noches_t.persona.to_numpy()
b_n = np.array([spearmanr(noches_t.real.to_numpy()[i], noches_t.estimada.to_numpy()[i])[0]
                for i in (indices(s_n, so) for so in sorteos(s_n))])
rho_n = spearmanr(noches_t.real, noches_t.estimada)[0]

fig, ax = plt.subplots(figsize=(3.6, 3.2))
ax.scatter(100 * noches_d.real, 100 * noches_d.estimada, s=8, color=GRIS, label="desarrollo")
for cat, g_ in noches_t.groupby("categoria"):
    ax.scatter(g_.real, g_.estimada, s=16, color=COLORES_CAT[cat], label=f"test, categoría {cat}")
for r_, f_ in noches_t.iterrows():
    ax.annotate(r_, (f_.real, f_.estimada), fontsize=5.5, xytext=(2, 2), textcoords="offset points")
ax.plot([0, 100], [0, 100], ":", color=GRIS, lw=0.8)
ax.set(xlabel="% real de minutos con apnea", ylabel="% marcado por el detector", xlim=(-3, 103), ylim=(-3, 103),
       title=f"Por noche · ρ = {rho_n:.2f}".replace(".", ","))
ax.legend(fontsize=6, loc="upper left")
fig.tight_layout()
guardar(fig, "05_por_noche")
print(f"ρ por noche: {rho_n:.3f} (IC 95 % {ic(b_n)[0]:.3f} a {ic(b_n)[1]:.3f})")
noches_t.sort_values("exactitud").round(2)

### 3.4 Dónde falla

La mejor y la peor noche de test según la exactitud: C minuto a minuto con el umbral, sobre la etiqueta real.

In [ ]:
orden = noches_t.sort_values("exactitud")
casos = [orden.index[-1], orden.index[0]]
fig, axs = plt.subplots(2, 1, figsize=(ANCHO, 3.6), sharex=True)
for ax, r_ in zip(axs, casos):
    g_ = T[T.record == r_]
    ev_ = evaluable(g_).to_numpy()
    ax.plot(g_.minute / 60, g_.C, color="black", lw=0.6)
    ax.scatter(g_.minute[ev_] / 60, g_.C[ev_], s=3, c=np.where(g_.label[ev_] == 1, APNEA, NORMAL), zorder=3)
    ax.axhline(P["umbral"], color="black", ls="--", lw=0.8)
    ax.set(ylabel="C", title=f"{r_} · categoría {CATEG[r_]} · exactitud {noches_t.exactitud[r_]:.2f} · "
                             f"apnea real {noches_t.real[r_]:.0f} %, marcada {noches_t.estimada[r_]:.0f} %".replace(".", ","))
axs[-1].set_xlabel("horas desde el inicio")
fig.tight_layout()
guardar(fig, "05_mejor_peor_noche")
print("Puntos naranjas: minutos con apnea según la etiqueta; azules: normales. Línea punteada: umbral.")

## 4. Desarrollo y test

Las mismas medidas del detector C en test, al lado de las de desarrollo. La columna de desarrollo es la de la **validación cruzada por personas** de `04` (secciones 7 y 8): es la estimación honesta de lo que se esperaba en personas nuevas. Sus AUC se calcularon de otra forma (promedio entre grupos; mediana entre noches), así que se comparan solo de forma aproximada.

In [ ]:
DEV_CV = {"sensibilidad": 0.814, "especificidad": 0.851, "exactitud": 0.836, "AUC (todos los minutos)": 0.907,
          "AUC dentro de la noche (media)": 0.924, "ρ por noche": 0.795, "% marcado en controles": 7.376}
comp = pd.DataFrame({"desarrollo (validación cruzada, 04)": DEV_CV, "test": punto["C"],
                     "IC 95 % (test)": {m_: "{:.3f} a {:.3f}".format(*ic(boot_f["C"][m_])) for m_ in DEV_CV}})
comp.round(3)

## 5. Resultados guardados

Se guardan en `docs/resultados_test.json` para el informe.

In [ ]:
RESULTADOS = {
    "registros": len(TEST), "personas": int(SUJETO[TEST].nunique()), "minutos_evaluados": int(len(et)),
    "predicciones": {k: {"valor": float(v["valor"]), "ic95": v["IC 95 %"], "se_cumple": bool(v["se cumple"])}
                     for k, v in veredicto.iterrows()},
    "H_se_sostiene": H_SOSTENIDA,
    "clasificacion_C": {k: {"valor": float(v), "ic95": "{:.3f} a {:.3f}".format(*ic(b3[k]))} for k, v in op.items()},
    "comparacion": {f_: {k: float(v) for k, v in punto[f_].items()} for f_ in COL},
    "diferencias": dif_feat.to_dict(orient="index"),
    "confianza": conf_tab.reset_index().to_dict(orient="records"),
    "rho_por_noche": {"valor": float(rho_n), "ic95": "{:.3f} a {:.3f}".format(*ic(b_n))},
    "umbrales_desarrollo": {k: float(v) for k, v in U.items()},
}
if not ENSAYO:
    (ROOT / "docs" / "resultados_test.json").write_text(json.dumps(RESULTADOS, indent=2, ensure_ascii=False, default=str))
print(json.dumps(RESULTADOS["predicciones"], indent=1, ensure_ascii=False))